<a href="https://colab.research.google.com/github/anya-44/basecamp-team-30/blob/main/clean_schools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import geopandas as gpd
df = pd.read_csv(
    "/content/schools_raw.csv",
    low_memory=False,
    encoding="cp1252"
)

df.head() # what is one row?


,URN,LA (code),LA (name),EstablishmentNumber,EstablishmentName,TypeOfEstablishment (code),TypeOfEstablishment (name),EstablishmentTypeGroup (code),EstablishmentTypeGroup (name),EstablishmentStatus (code),...,QABName (code),QABName (name),EstablishmentAccredited (code),EstablishmentAccredited (name),QABReport,CHNumber,MSOA (code),LSOA (code),FSM,AccreditationExpiryDate
0,100000,201,City of London,3614.0,The Aldgate School,2,Voluntary aided school,4,Local authority maintained schools,1,...,0,Not applicable,0,Not applicable,NaN,NaN,E02000001,E01032739,52.0,NaN
1,100001,201,City of London,6005.0,City of London School for Girls,11,Other independent school,3,Independent schools,1,...,0,Not applicable,0,Not applicable,NaN,NaN,E02000001,E01000002,NaN,NaN
2,100002,201,City of London,6006.0,St Paul's Cathedral School,11,Other independent school,3,Independent schools,1,...,0,Not applicable,0,Not applicable,NaN,NaN,E02000001,E01032739,NaN,NaN
3,100003,201,City of London,6007.0,City of London School,11,Other independent school,3,Independent schools,1,...,0,Not applicable,0,Not applicable,NaN,NaN,E02000001,E01032739,NaN,NaN
4,100004,202,Camden,1045.0,Sherborne Nursery School,15,Local authority nursery school,4,Local authority maintained schools,2,...,0,Not applicable,0,Not applicable,NaN,NaN,E02000172,E01000901,NaN,NaN


In [7]:
# This cell shows that there are a lot of primary schools with missing opening
# dates in the unfiltered dataset so we must filter dates carefully
# (dates are filtered later on)

# Define primary schools from the filtered dataset df
primary = df.copy()

# Start with London primary schools
test = primary.copy()

test["OpenDate"] = pd.to_datetime(
    test["OpenDate"], dayfirst=True, errors="coerce"
)

test["CloseDate"] = pd.to_datetime(
    test["CloseDate"], dayfirst=True, errors="coerce"
)

# Currently open schools with opening dates after 2020
opened_later = test[
    (test["EstablishmentStatus (name)"] == "Open") &
    (test["OpenDate"] > "2021-01-01")
]

# Get all open primary schools
open_primary = test[test["EstablishmentStatus (name)"] == "Open"]

print("Currently open primary schools:", len(open_primary))
print("Opened after start of 2021:", len(opened_later))

# Check missing opening dates
print("Missing opening dates:",
      test["OpenDate"].isna().sum())

print("This is processed later.")


Currently open primary schools: 27159
Opened after start of 2021: 3723
Missing opening dates: 30039
This is processed later.


In [8]:
df.info() # column types and blanks


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52622 entries, 0 to 52621
Columns: 135 entries, URN to AccreditationExpiryDate
dtypes: float64(28), int64(18), object(89)
memory usage: 54.2+ MB


In [9]:
df.describe() # impossible numbers


,URN,LA (code),EstablishmentNumber,TypeOfEstablishment (code),EstablishmentTypeGroup (code),EstablishmentStatus (code),ReasonEstablishmentOpened (code),ReasonEstablishmentClosed (code),PhaseOfEducation (code),StatutoryLowAge,...,Northing,SENStat,SENNoStat,PreviousLA (code),PreviousEstablishmentNumber,UPRN,QABName (code),EstablishmentAccredited (code),CHNumber,FSM
count,52622.000000,52622.000000,52486.000000,52622.000000,52622.000000,52622.000000,52622.000000,52622.000000,52622.000000,48629.000000,...,50842.000000,4017.000000,3688.000000,52622.000000,8367.000000,3.990000e+04,52622.000000,52622.000000,1.400000e+01,28300.000000
mean,138554.985253,684.906845,3575.923732,13.048611,5.628520,1.485690,4.063833,29.041352,1.880487,5.613050,...,281087.662149,12.320637,30.991866,969.290544,3681.394287,6.175790e+10,0.000323,0.000323,9.065504e+06,88.297915
std,60355.887741,262.888927,1750.259419,13.834019,2.702277,0.504568,12.558699,42.784269,1.370534,3.537522,...,126732.195769,26.006296,62.394790,53.232674,1594.571647,6.841019e+10,0.017971,0.017971,2.953921e+06,106.033394
min,100000.000000,0.000000,15.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,1000.000000,1.084000e+03,0.000000,0.000000,4.236028e+06,0.000000
25%,113166.250000,371.000000,2147.000000,1.000000,4.000000,1.000000,0.000000,0.000000,1.000000,3.000000,...,178447.750000,0.000000,0.000000,924.000000,2329.000000,5.300044e+09,0.000000,0.000000,7.606934e+06,21.000000
50%,126330.500000,838.000000,3100.000000,5.000000,4.000000,1.000000,0.000000,4.000000,2.000000,4.000000,...,272508.000000,2.000000,5.000000,999.000000,3173.000000,1.009026e+10,0.000000,0.000000,9.134706e+06,53.000000
75%,140348.750000,888.000000,4321.000000,28.000000,9.000000,2.000000,6.000000,99.000000,2.000000,7.000000,...,389797.750000,11.000000,30.000000,999.000000,4624.500000,1.000714e+11,0.000000,0.000000,1.155595e+07,112.000000
max,402495.000000,943.000000,9921.000000,57.000000,13.000000,4.000000,99.000000,99.000000,7.000000,19.000000,...,811856.000000,402.000000,771.000000,999.000000,9901.000000,4.220001e+11,1.000000,1.000000,1.319900e+07,1148.000000


In [10]:
df.duplicated().sum() # repeated rows

np.int64(0)

In [11]:
# Only keep London schools
before = len(df)
df = df[
    df["DistrictAdministrative (code)"]
    .astype(str)
    .str.startswith("E09")
]
after = len(df)
print(f"Schools remaining in London: {after}")
print(f"Schools removed: {before - after}")

# Check for missing or zero coordinates
missing_coords = df["Easting"].isna() | df["Northing"].isna() | (df["Easting"] == 0) | (df["Northing"] == 0)
print(f"Schools with missing or zero coordinates: {missing_coords.sum()}")

# Check for impossible locations (outside London box: Easting 500k-565k, Northing 155k-205k)
out_of_bounds = (~missing_coords) & (
    (df["Easting"] < 500000) | (df["Easting"] > 565000) |
    (df["Northing"] < 155000) | (df["Northing"] > 205000)
)
print(f"Schools outside London coordinate bounds: {out_of_bounds.sum()}")

Schools remaining in London: 5827
Schools removed: 46795
Schools with missing or zero coordinates: 22
Schools outside London coordinate bounds: 3


In [12]:
# Count schools in each category
codes = {
    2: "Primary",
    3: "Middle deemed primary",
    7: "All-through"
}

counts = (
    df[df["PhaseOfEducation (code)"].isin(codes.keys())]
    ["PhaseOfEducation (code)"]
    .value_counts()
    .reindex(codes.keys(), fill_value=0)
)

# Display results
for code, count in counts.items():
    print(f"{codes[code]}: {count:,}")

print(f"\nTotal: {counts.sum():,}")


df = df[df["PhaseOfEducation (code)"] == 2] # keeps only primary schools
print("All non primaries removed")

Primary: 3,046
Middle deemed primary: 34
All-through: 52

Total: 3,132
All non primaries removed


In [13]:
# Define research period
start_date = pd.Timestamp("2021-01-01")
end_date = pd.Timestamp("2025-12-31")

# Convert dates into datetime format
df["OpenDate"] = pd.to_datetime(
    df["OpenDate"], dayfirst=True, errors="coerce"
)

df["CloseDate"] = pd.to_datetime(
    df["CloseDate"], dayfirst=True, errors="coerce"
)

# Keep schools that:
# - Opened before or on 1 January 2021, OR have a missing opening date
# - Have no closing date, OR closed after 31 December 2025
# - Exclude schools that opened after 1 January 2021.
# - Exclude schools that closed during 2021–2025.


df = df[
    (
        (df["OpenDate"].isna()) |
        (df["OpenDate"] <= start_date)
    ) &
    (
        (df["CloseDate"].isna()) |
        (df["CloseDate"] > end_date)
    )
]

# Reset index
df = df.reset_index(drop=True)

print("Schools retained:", len(df))
print("Schools with missing opening dates:",
      df["OpenDate"].isna().sum())


Schools retained: 1672
Schools with missing opening dates: 1006


In [14]:
# Check and log special & independent schools inside our dataset
if "EstablishmentTypeGroup (name)" in df.columns:
    display(df["EstablishmentTypeGroup (name)"].value_counts())

# Columns of interest subsetting
required_cols = [
    "URN",
    "EstablishmentName",
    "Easting",
    "Northing",
    "DistrictAdministrative (code)",
    "LSOA (code)",
    "TypeOfEstablishment (name)",
    "StatutoryLowAge",
    "StatutoryHighAge"
]

# Keep only available columns from the list
available_cols = [col for col in required_cols if col in df.columns]
df = df[available_cols].copy()



,count
EstablishmentTypeGroup (name),
Local authority maintained schools,1165
Academies,442
Free Schools,65


In [15]:
df = df.reset_index(drop=True)

print(df.info())

display(df.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1672 entries, 0 to 1671
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   URN                            1672 non-null   int64  
 1   EstablishmentName              1672 non-null   object 
 2   Easting                        1672 non-null   float64
 3   Northing                       1672 non-null   float64
 4   DistrictAdministrative (code)  1672 non-null   object 
 5   LSOA (code)                    1672 non-null   object 
 6   TypeOfEstablishment (name)     1672 non-null   object 
 7   StatutoryLowAge                1672 non-null   float64
 8   StatutoryHighAge               1672 non-null   float64
dtypes: float64(4), int64(1), object(4)
memory usage: 117.7+ KB
None


,URN,EstablishmentName,Easting,Northing,DistrictAdministrative (code),LSOA (code),TypeOfEstablishment (name),StatutoryLowAge,StatutoryHighAge
0,100000,The Aldgate School,533498.0,181201.0,E09000001,E01032739,Voluntary aided school,3.0,11.0
1,100008,Argyle Primary School,530238.0,182761.0,E09000007,E01000941,Community school,3.0,11.0
2,100009,West Hampstead Primary School,524888.0,185067.0,E09000007,E01000873,Community school,3.0,11.0
3,100010,Brecknock Primary School,529912.0,184835.0,E09000007,E01000869,Community school,2.0,11.0
4,100011,Brookfield Primary School,528706.0,186594.0,E09000007,E01000908,Community school,2.0,11.0
